# Chấm submission trên máy local

Mở notebook bằng VS Code/Jupyter và chọn kernel `.venv` Python 3.13.
Cài `requirements-notebook.txt` vào `.venv` trước khi chạy.
Chạy hai cell đầu để hiện form, chọn track/mode/ID/config, điền owner,
rồi chạy cell chấm bên dưới. Khi mở form mới, không có ID/config được chọn sẵn. Artifact được đọc trực tiếp
trong project, kết quả lưu vào `results/<TRACK>/<ID>/<CONFIG_NAME>/<MODE>/<run_id>/`.
Dev có timeout 300 giây; full chạy theo mặc định chính thức. Không cần upload/giải nén results.


In [ ]:
from pathlib import Path
import sys, importlib.metadata

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'src/common/local_grade.py').is_file() and (p/'requirements.txt').is_file()), None)
if ROOT is None:
    raise RuntimeError('Mở notebook từ thư mục project hoặc notebooks/')
if sys.version_info[:2] != (3, 13) or Path(sys.prefix).resolve() != (ROOT/'.venv').resolve():
    raise RuntimeError('Chọn kernel Python 3.13 của .venv trong project')
EXPECTED_PACKAGES = {'kickmix': '1.0.0', 'numpy': '2.5.4', 'scipy': '1.18.1'}
for package, version in EXPECTED_PACKAGES.items():
    if importlib.metadata.version(package) != version:
        raise RuntimeError(f'Cài lại requirements.txt bằng .venv: {package} cần {version}')
print('Project:', ROOT)
print('Python:', sys.executable)


In [ ]:
import re
import html
import ipywidgets as widgets
from IPython.display import display

if 'grading_form' in globals():
    grading_form.close()
style = {'description_width': '130px'}
track_input = widgets.Dropdown(options=[('Chọn track', None), ('invsqrt', 'invsqrt'), ('dqi', 'dqi'), ('dfthcsos', 'dfthcsos')], description='TRACK', style=style)
mode_input = widgets.Dropdown(options=['dev_5min', 'smoke', 'full'], description='MODE', style=style)
id_input = widgets.Dropdown(options=[('Chọn ID', None)], description='EXPERIMENT_ID', style=style, disabled=True)
config_input = widgets.Dropdown(options=[('Chọn config', None)], description='CONFIG_NAME', style=style, disabled=True)
owner_input = widgets.Text(value='', placeholder='GitHub username của bạn', description='OWNER', style=style)
refresh_input = widgets.Button(description='Làm mới danh sách')
selection_info = widgets.HTML()

# Chỉ liệt kê ID đúng track và config có cả JSON lẫn artifact đã compile.
def update_preview(change=None):
    track, exp, config = track_input.value, id_input.value, config_input.value
    if not track or not exp or not config:
        selection_info.value = 'Chọn TRACK → EXPERIMENT_ID → CONFIG_NAME. Config chỉ hiện khi đã có JSON và artifact.'
        return
    ext = '.zip' if track == 'dfthcsos' else '.kmb'
    artifact = ROOT/'artifacts'/track/exp/(config+ext)
    destination = ROOT/'results'/track/exp/config/mode_input.value
    selection_info.value = '<b>Artifact:</b> '+html.escape(str(artifact))+'<br><b>Results:</b> '+html.escape(str(destination))+'/&lt;run_id&gt;'
    if not owner_input.value.strip():
        selection_info.value += '<br>Điền OWNER trước khi chạy cell chấm.'

def update_configs(change=None):
    track, exp = track_input.value, id_input.value
    names = []
    if track and exp:
        ext = '.zip' if track == 'dfthcsos' else '.kmb'
        names = sorted(p.stem for p in (ROOT/'src'/track/exp/'configs').glob('*.json')
                       if re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]*', p.stem)
                       and (ROOT/'artifacts'/track/exp/(p.stem+ext)).is_file())
    config_input.options = [('Chọn config', None)] + [(name, name) for name in names]
    config_input.value = None
    config_input.disabled = not names
    update_preview()

def update_ids(change=None):
    track = track_input.value
    ids = []
    if track:
        prefix = {'invsqrt': 'INV', 'dqi': 'DQI', 'dfthcsos': 'DFT'}[track]
        ids = sorted(p.name for p in (ROOT/'src'/track).iterdir()
                     if p.is_dir() and re.fullmatch(prefix+r'-(BASELINE|[1-9][0-9]*)', p.name)
                     and (p/'DESCRIPTION.md').is_file())
    id_input.options = [('Chọn ID', None)] + [(name, name) for name in ids]
    id_input.value = None
    id_input.disabled = not ids
    update_configs()

track_input.observe(update_ids, names='value')
id_input.observe(update_configs, names='value')
config_input.observe(update_preview, names='value')
mode_input.observe(update_preview, names='value')
owner_input.observe(update_preview, names='value')
refresh_input.on_click(update_ids)
grading_form = widgets.VBox([track_input, mode_input, id_input, config_input, owner_input, refresh_input, selection_info])
update_preview()
display(grading_form)


In [ ]:
# Đọc lựa chọn hiện tại trực tiếp từ dropdown, không dùng biến của lượt trước.
TRACK, MODE = track_input.value, mode_input.value
EXPERIMENT_ID, CONFIG_NAME = id_input.value, config_input.value
OWNER = owner_input.value.strip()
if not all([TRACK, MODE, EXPERIMENT_ID, CONFIG_NAME, OWNER]):
    raise ValueError('Chọn đủ track, ID, config và điền OWNER ở form trước khi chấm')
import importlib.util
spec = importlib.util.spec_from_file_location('local_grade', ROOT/'src/common/local_grade.py')
grader = importlib.util.module_from_spec(spec)
spec.loader.exec_module(grader)
OUT = grader.grade(ROOT, TRACK, MODE, EXPERIMENT_ID, OWNER, CONFIG_NAME)


Kết quả đã ở đúng thư mục local, gồm config chấm, metadata, stdout/stderr và
verifier.json khi có output JSON. Provenance được nối từ build_only nếu tìm được
đúng một record khớp hash; nếu thiếu/mơ hồ, agent cần xử lý review.json trước PR.
Báo agent đường dẫn Saved để phân tích và cập nhật tracking. BUILT/dev PASS không
thay cho full PASS. Đổi MODE/CONFIG_NAME rồi chạy lại sẽ tạo run mới.
